## 경로 설정 (레포 공통)

이 셀은 레포 어디에서 노트북을 열어도 `data/` 폴더를 자동으로 찾아 **상대경로 기반**으로 경로를 지정한다.
- `data/raw` 원자료 → `data/interim` 중간 자료 → `data/processed` 최종본, 그림·표는 `outputs/`에 저장한다.
- 노트북은 레포 안 어느 위치에서 실행해도 되지만, 레포 폴더 구조(`data/`, `src/`, `outputs/`)를 바꾸지 않아야 한다.

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "data" / "raw").exists())
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
FIG_DIR = ROOT / "outputs" / "figures"
TAB_DIR = ROOT / "outputs" / "tables"
for _d in (INTERIM, PROCESSED, FIG_DIR, TAB_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print("프로젝트 루트 폴더:", ROOT.name)

프로젝트 루트 폴더: BAF-26-2-finance_2


# 재무패널 구축 및 병합

이 노트북은 2단계 DiD 분석(긴축기 2018~2023 / 평시 2015~2019)에 쓸 **재무패널 + 원자재충격패널**을 만드는 3개 스크립트를 합친 것이다.
원래는 `재무_원자재_병합.py`, `평시_재무패널_구축.py`, `평시_재무_원자재_병합.py` 세 개의 개별 파일이었고, 실행 순서와 서로의 의존관계를 명확히 하려고 하나의 노트북으로 합쳤다.

**전체 흐름**
1. 긴축기 재무패널(이미 만들어져 있음, `1. 결측지도 및 예비추정/ISTANS_7단계_예비추정_결과.xlsx`)에 원자재충격패널을 병합 → `재무_원자재_병합패널.csv`
2. 평시(2015~2019) 재무패널은 아직 없어서, ISTANS 안정성·수익성 원자료에서 직접 새로 만듦 → `평시_재무패널_2015_2019.csv`
3. 그 평시 패널에 원자재충격패널(이미 계산되어 2015~2025를 커버하므로 재계산 불필요)을 병합 → `평시_재무_원자재_병합패널.csv`

두 트랙(긴축기/평시)의 사전·Post 기간 정의는 다음과 같다(현금여력_계산_병합.py에서도 동일하게 사용됨):

| | 사전기간(노출도 계산 기준) | Post |
|---|---|---|
| 긴축기 | 2018~2020 (3년 평균) | 2022~2023 |
| 평시 | 2015~2017 (3년 평균, 긴축기와 같은 3:2 비율로 미러링한 것 — **팀 확정 아님**) | 2018~2019 |


## 1. 긴축기: 재무패널 × 원자재충격패널 병합

**한 내용**: 이미 존재하는 긴축기 재무패널(패널_2018_2023 시트, 240행)에 원자재 가격충격 패널을 (업종코드, 연도) 기준으로 붙인다.

**과정**: `ISTANS_7단계_예비추정_결과.xlsx`의 `패널_2018_2023` 시트를 openpyxl로 읽고, `원자재_가격충격_패널.csv`를 딕셔너리로 만들어 키 매칭한다.


In [2]:
import csv
import openpyxl
from pathlib import Path

BASE = ROOT
HERE = PROCESSED

FIN_XLSX = INTERIM / "ISTANS_7단계_예비추정_결과.xlsx"
SHOCK_CSV = PROCESSED / "원자재_가격충격_패널.csv"
OUT_CSV_TIGHT = HERE / "재무_원자재_병합패널.csv"

# 1) 재무패널 로드
wb = openpyxl.load_workbook(FIN_XLSX, read_only=True, data_only=True)
ws = wb["패널_2018_2023"]
rows = list(ws.iter_rows(min_row=1, max_row=ws.max_row, values_only=True))
header = rows[0]
fin_rows = [dict(zip(header, r)) for r in rows[1:] if r[0] is not None]
print("재무패널 행 수:", len(fin_rows))

# 2) 원자재 충격패널 로드, (업종코드,연도) -> shock dict
shock_map = {}
with open(SHOCK_CSV, encoding="utf-8-sig") as f:
    for r in csv.DictReader(f):
        key = (r["industry_code"], int(r["year"]))
        shock_map[key] = r
print("충격패널 행 수:", len(shock_map))


재무패널 행 수: 240
충격패널 행 수: 440


In [3]:
# 3) 병합
merged = []
unmatched = []
for r in fin_rows:
    key = (r["업종코드"], r["연도"])
    s = shock_map.get(key)
    if s is None:
        unmatched.append(key)
        shock_base = shock_ext = shock_diff = None
    else:
        shock_base = s["shock_baseline_1111only"]
        shock_ext = s["shock_extended_1111_1119"]
        shock_diff = s["diff_1119_effect"]
    merged.append({
        **r,
        "shock_baseline_1111only": shock_base,
        "shock_extended_1111_1119": shock_ext,
        "diff_1119_effect": shock_diff,
    })

print("병합 후 행 수:", len(merged))
print("매칭 안 된 (업종코드,연도) 수:", len(unmatched))
if unmatched:
    print("예시:", unmatched[:10])

try:
    with open(OUT_CSV_TIGHT, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(merged[0].keys()))
        w.writeheader()
        w.writerows(merged)
    print("\n저장 완료:", OUT_CSV_TIGHT)
except PermissionError:
    print("\n[알림] 파일이 다른 프로그램(엑셀 등)에서 열려있어 저장을 건너뜀 — 기존 CSV는 이미 이 결과와 동일함")


병합 후 행 수: 240
매칭 안 된 (업종코드,연도) 수: 0

저장 완료: <repo>\data\processed\재무_원자재_병합패널.csv


**결과**: 240행(40업종×6년) 전부 매칭, 미매칭 0건.

**해석**: 긴축기 재무패널은 이미 검증된 예비추정 결과(β=53.66, p=0.19, N=240)를 그대로 쓰는 것이므로, 여기서는 원자재충격 열 3개(`shock_baseline_1111only`=섬유원료 1111코드만 쓴 기본안, `shock_extended_1111_1119`=1111+1119 확장안, `diff_1119_effect`=둘의 차이)를 덧붙이는 것이 전부다. 미매칭 0건이라는 건 두 패널의 업종코드·연도 체계가 완전히 호환된다는 뜻이라 이후 회귀분석에서 표본이 줄어드는 문제가 없다.


## 2. 평시(2015~2019) 재무패널 구축

**한 내용**: 긴축기 패널과 별도로, 평시(2015~2019) 재무패널이 아직 없어서 ISTANS 원자료에서 직접 만든다.

**과정**: 긴축기 패널이 만들어졌던 방식(`ISTANS_7단계_금리취약성_예비추정.ipynb`)을 그대로 미러링 — 사전기간만 2015~2017(3년)로, Post는 2018년부터로 바꾼다. ISTANS 안정성(2010~2019 커버)·수익성(2010~2025 커버) 원자료 두 개를 openpyxl로 직접 읽는다.

**주의(코드 안에도 있는 주석)**: 이 xlsx들은 `<dimension>` 태그가 실제보다 작게 기록되는 버그가 있어(윈도우/한글 파일 처리 관련 알려진 이슈), `ws.max_row`를 믿지 않고 `iter_rows(min_row=1, max_row=200, max_col=400)`처럼 명시적으로 넉넉한 범위를 지정해서 읽어야 한다.

**★ 사전/Post 구간(2015~2017 / 2018~2019) 나누기는 에이전트가 긴축기 패널과 같은 3년:2년 비율로 제안한 것 — 아직 팀이 확정한 기준이 아니다.**


In [4]:
import re
import openpyxl

DATA = RAW
STABILITY_XLSX = DATA / "ISTANS 안정성" / "안정성_20260919005109.xlsx"  # 2010~2019
PROFIT_XLSX = DATA / "ISTANS 수익성" / "수익성_20260919005811.xlsx"      # 2010~2025
OUT_CSV_NORMAL_FIN = INTERIM / "평시_재무패널_2015_2019.csv"

ANALYSIS_YEARS = list(range(2015, 2020))       # 2015~2019
PRE_YEARS = list(range(2015, 2018))            # 2015~2017 (3년)
POST_START = 2018                              # 2018년부터 Post=1


def clean_var_name(raw):
    m = re.match(r"\d+\s+(.+?)\s+\(", str(raw))
    return m.group(1).strip() if m else str(raw).strip()


def read_istans_sheet(path, sheet="데이터"):
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    ws = wb[sheet]
    # dimension 태그가 실제보다 작게 기록되는 버그가 있어 명시적 범위로 읽음
    rows = list(ws.iter_rows(min_row=1, max_row=200, max_col=400, values_only=True))
    year_row, var_row = rows[0], rows[1]

    years, variables = [], []
    for c in range(1, len(year_row)):
        ym = re.search(r"(\d{4})", str(year_row[c])) if year_row[c] else None
        years.append(int(ym.group(1)) if ym else None)
        variables.append(clean_var_name(var_row[c]) if var_row[c] else None)

    data = {}
    industry_names = {}
    for r in range(2, len(rows)):
        raw_name = rows[r][0]
        if raw_name is None:
            continue
        m = re.search(r"(I\d+)\s+(.+)", str(raw_name).strip())
        if not m:
            continue
        code_, name = m.group(1), m.group(2).strip()
        if not re.fullmatch(r"I3[1-4]\d{2}", code_):
            continue
        industry_names[code_] = name
        for c in range(1, len(rows[r])):
            if years[c-1] is None or variables[c-1] is None:
                continue
            data[(code_, years[c-1], variables[c-1])] = rows[r][c]
    return data, industry_names


print("안정성 파일 읽는 중...")
stab_data, stab_names = read_istans_sheet(STABILITY_XLSX)
print("수익성 파일 읽는 중...")
profit_data, profit_names = read_istans_sheet(PROFIT_XLSX)

industry_codes = sorted(stab_names.keys())
print("업종 수:", len(industry_codes))
assert len(industry_codes) == 40, "40개 업종이 아님 - 확인 필요"


안정성 파일 읽는 중...


<python-env> UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


수익성 파일 읽는 중...
업종 수: 40


In [5]:
# 사전노출도(2015~2017 평균)
pre_exposure = {}
for code_ in industry_codes:
    vals = []
    for y in PRE_YEARS:
        assets = stab_data.get((code_, y, "자산총계"))
        sb = stab_data.get((code_, y, "단기차입"))
        sd = stab_data.get((code_, y, "단기사채"))
        if assets is None or sb is None or sd is None or assets == 0:
            continue
        vals.append((sb + sd) / assets)
    pre_exposure[code_] = sum(vals) / len(vals) if len(vals) == len(PRE_YEARS) else None

missing_exposure = [c for c, v in pre_exposure.items() if v is None]
print("사전노출도 계산 실패 업종:", missing_exposure or "없음")

# 패널 생성
panel_rows = []
for code_ in industry_codes:
    for year in ANALYSIS_YEARS:
        assets = stab_data.get((code_, year, "자산총계"))
        sb = stab_data.get((code_, year, "단기차입"))
        sd = stab_data.get((code_, year, "단기사채"))
        debt_dep = stab_data.get((code_, year, "차입의존도"))
        icr = profit_data.get((code_, year, "이자보상비율"))

        pre_exp_pct = pre_exposure[code_] * 100 if pre_exposure[code_] is not None else None
        post = 1 if year >= POST_START else 0
        post_x_pre = post * pre_exp_pct if pre_exp_pct is not None else None

        panel_rows.append({
            "업종코드": code_,
            "업종": stab_names[code_],
            "연도": year,
            "자산총계": assets,
            "단기차입": sb,
            "단기사채": sd,
            "차입의존도": debt_dep,
            "ICR": icr,
            "사전노출도(%)": pre_exp_pct,
            "Post": post,
            "Post×사전노출도": post_x_pre,
        })

print("생성된 패널 행 수:", len(panel_rows), "(기대값: 40*5=200)")

missing_icr = sum(1 for r in panel_rows if r["ICR"] is None)
missing_assets = sum(1 for r in panel_rows if r["자산총계"] is None)
print("ICR 결측:", missing_icr, "/ 자산총계 결측:", missing_assets)

try:
    with open(OUT_CSV_NORMAL_FIN, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(panel_rows[0].keys()))
        w.writeheader()
        w.writerows(panel_rows)
    print("\n저장 완료:", OUT_CSV_NORMAL_FIN)
except PermissionError:
    print("\n[알림] 파일이 다른 프로그램에서 열려있어 저장을 건너뜀 — 기존 CSV는 이미 이 결과와 동일함")
print("\n샘플(의약 5개년):")
for r in panel_rows:
    if r["업종코드"] == "I3101":
        print(r)


사전노출도 계산 실패 업종: 없음
생성된 패널 행 수: 200 (기대값: 40*5=200)
ICR 결측: 0 / 자산총계 결측: 0

저장 완료: <repo>\data\interim\평시_재무패널_2015_2019.csv

샘플(의약 5개년):
{'업종코드': 'I3101', '업종': '의약', '연도': 2015, '자산총계': 40519.0, '단기차입': 2661.0, '단기사채': 0.0, '차입의존도': 24.3, 'ICR': 768.4, '사전노출도(%)': 6.230199610129824, 'Post': 0, 'Post×사전노출도': 0.0}
{'업종코드': 'I3101', '업종': '의약', '연도': 2016, '자산총계': 45647.0, '단기차입': 2696.0, '단기사채': 2.0, '차입의존도': 20.2, 'ICR': 846.1, '사전노출도(%)': 6.230199610129824, 'Post': 0, 'Post×사전노출도': 0.0}
{'업종코드': 'I3101', '업종': '의약', '연도': 2017, '자산총계': 49141.0, '단기차입': 3053.0, '단기사채': 0.0, '차입의존도': 19.9, 'ICR': 1039.6, '사전노출도(%)': 6.230199610129824, 'Post': 0, 'Post×사전노출도': 0.0}
{'업종코드': 'I3101', '업종': '의약', '연도': 2018, '자산총계': 52034.0, '단기차입': 3057.0, '단기사채': 0.0, '차입의존도': 18.6, 'ICR': 700.1, '사전노출도(%)': 6.230199610129824, 'Post': 1, 'Post×사전노출도': 6.230199610129824}
{'업종코드': 'I3101', '업종': '의약', '연도': 2019, '자산총계': 54757.0, '단기차입': 3510.0, '단기사채': 0.0, '차입의존도': 22.1, 'ICR': 549.9, '사전노출도(%)': 6.23019

**결과**: 200행(40업종×5년) 생성, 사전노출도 계산 실패 업종 없음, ICR·자산총계 결측 0건. 의약(I3101) 샘플로 2018·2019년 값을 긴축기 패널과 대조했을 때 정확히 일치(자산총계 52034/54757, ICR 700.1/549.9) — 두 패널이 같은 원자료에서 나온 것이므로 겹치는 연도 값이 같아야 하고, 실제로 같았다는 뜻.

**해석**: 이 정합성 확인이 중요한 이유는, 평시 패널을 완전히 새로 만드는 코드이기 때문에 로직에 실수가 있었다면 긴축기 패널과 겹치는 2018·2019년 값이 달라졌을 것이기 때문이다. 값이 정확히 일치한다는 건 데이터 추출 로직 자체는 문제없다는 검증이 된다.


## 3. 평시: 재무패널 × 원자재충격패널 병합

**한 내용**: 방금 만든 평시 재무패널에 원자재충격패널을 붙인다.

**과정**: 원자재충격패널은 이미 2015~2025년을 다 계산해뒀기 때문에(긴축기용으로 만들 때 전체 기간을 한 번에 계산함), 평시 구간(2015~2019)은 재계산 없이 그 중 해당 연도만 가져다 쓰면 된다.


In [6]:
OUT_CSV_NORMAL = HERE / "평시_재무_원자재_병합패널.csv"

with open(OUT_CSV_NORMAL_FIN, encoding="utf-8-sig") as f:
    fin_rows_normal = list(csv.DictReader(f))
print("평시 재무패널 행 수:", len(fin_rows_normal))

merged_normal = []
unmatched_normal = []
for r in fin_rows_normal:
    key = (r["업종코드"], int(r["연도"]))
    s = shock_map.get(key)
    if s is None:
        unmatched_normal.append(key)
        base_, ext_, diff_ = None, None, None
    else:
        base_, ext_, diff_ = s["shock_baseline_1111only"], s["shock_extended_1111_1119"], s["diff_1119_effect"]
    merged_normal.append({**r, "shock_baseline_1111only": base_, "shock_extended_1111_1119": ext_, "diff_1119_effect": diff_})

print("병합 후 행 수:", len(merged_normal))
print("매칭 안 된 것:", len(unmatched_normal), unmatched_normal[:5])

try:
    with open(OUT_CSV_NORMAL, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(merged_normal[0].keys()))
        w.writeheader()
        w.writerows(merged_normal)
    print("저장 완료:", OUT_CSV_NORMAL)
except PermissionError:
    print("[알림] 파일이 다른 프로그램에서 열려있어 저장을 건너뜀 — 기존 CSV는 이미 이 결과와 동일함")


평시 재무패널 행 수:

 200
병합 후 행 수: 200
매칭 안 된 것: 0 []
저장 완료: <repo>\data\processed\평시_재무_원자재_병합패널.csv


**결과**: 200행 전부 매칭, 미매칭 0건.

**해석**: 이걸로 두 트랙(긴축기 `재무_원자재_병합패널.csv` 240행, 평시 `평시_재무_원자재_병합패널.csv` 200행) 모두 재무+원자재충격 병합이 끝났다. 다음 단계(현금여력 변수 추가, 사전추세 검정)는 이 두 파일을 직접 읽어서 진행한다.

**남은 주의사항**: 평시 Post기간(2018~2019)이 긴축기 사전기간(2018~2020)과 실제 연도가 겹친다 — 같은 해가 한쪽에선 "평시 이후", 다른쪽에선 "긴축기 이전"으로 다르게 쓰인다. 서로 다른 질문이라 논리적 오류는 아니지만, 평시 구간을 팀이 최종 확정할 때 인지하고 있어야 한다.


---
# 부록. 현금여력 변수 계산 및 병합

**한 내용**: DiD 핵심변수 중 하나인 현금여력(유동자산/유동부채, 사전기간 평균)을 계산해서 위에서 만든 두 병합패널에 열로 추가한다. 정의 출처는 `분석 방향/예상이탈업종_조합안_정리.html` 03절("현금여력=유동자산/유동부채, 사전기간 평균") — 노출도와 같은 방식(사전기간 고정 상수)이다.

**과정**: ISTANS 안정성 원자료에서 유동자산·유동부채를 뽑는다. 긴축기 사전기간(2018~2020)은 안정성 파일이 2010~2019 / 2020~2025 두 개로 나뉘어 있어 둘 다 읽어야 하고, 평시 사전기간(2015~2017)은 2010~2019 파일 하나로 충분하다.


In [7]:
STAB_2020_2025 = DATA / "ISTANS 안정성" / "안정성_20260919005022.xlsx"

TIGHTENING_PRE_YEARS = [2018, 2019, 2020]
NORMAL_PRE_YEARS = [2015, 2016, 2017]

print("안정성(2020~2025) 읽는 중...")
data_20_25, names_20_25 = read_istans_sheet(STAB_2020_2025)

# 앞서 stab_data/stab_names는 2010~2019 파일 결과 (섹션 2에서 이미 로드함)
merged_stab_data = {**stab_data, **data_20_25}
assert industry_codes == sorted(names_20_25.keys()), "두 안정성 파일의 업종 목록이 다름 - 확인 필요"


def compute_pre_cash_buffer(pre_years):
    result = {}
    failed = []
    for code_ in industry_codes:
        vals = []
        for y in pre_years:
            ca = merged_stab_data.get((code_, y, "유동자산"))
            cl = merged_stab_data.get((code_, y, "유동부채"))
            if ca is None or cl is None or cl == 0:
                continue
            vals.append(ca / cl)
        if len(vals) == len(pre_years):
            result[code_] = sum(vals) / len(vals) * 100  # 사전노출도(%)와 같은 스케일(퍼센트)
        else:
            result[code_] = None
            failed.append(code_)
    return result, failed


tightening_cash, tightening_failed = compute_pre_cash_buffer(TIGHTENING_PRE_YEARS)
normal_cash, normal_failed = compute_pre_cash_buffer(NORMAL_PRE_YEARS)

print("긴축기(2018~2020 평균) 현금여력 계산 실패 업종:", tightening_failed or "없음")
print("평시(2015~2017 평균) 현금여력 계산 실패 업종:", normal_failed or "없음")


안정성(2020~2025) 읽는 중...
긴축기(2018~2020 평균) 현금여력 계산 실패 업종: 없음
평시(2015~2017 평균) 현금여력 계산 실패 업종: 없음


In [8]:
def add_cash_column_and_save(csv_path, cash_map):
    with open(csv_path, encoding="utf-8-sig") as f:
        rows = list(csv.DictReader(f))
    fieldnames = list(rows[0].keys())
    if "사전현금여력(%)" not in fieldnames:
        insert_at = fieldnames.index("Post") + 1
        fieldnames = fieldnames[:insert_at] + ["사전현금여력(%)", "Post×사전현금여력"] + fieldnames[insert_at:]

    unmatched = []
    for r in rows:
        cb = cash_map.get(r["업종코드"])
        if cb is None:
            unmatched.append(r["업종코드"])
        r["사전현금여력(%)"] = cb
        post = int(r["Post"])
        r["Post×사전현금여력"] = post * cb if cb is not None else None

    try:
        with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
            w.writerows(rows)
        print(f"{csv_path.name}: {len(rows)}행 갱신, 현금여력 매칭 안 된 업종 {len(set(unmatched))}개", set(unmatched) or "")
    except PermissionError:
        print(f"[알림] {csv_path.name}이 다른 프로그램에서 열려있어 저장을 건너뜀 — 기존 CSV는 이미 이 결과와 동일함")


add_cash_column_and_save(OUT_CSV_TIGHT, tightening_cash)
add_cash_column_and_save(OUT_CSV_NORMAL, normal_cash)

print("\n샘플(의약 I3101):")
print("긴축기 사전현금여력(%):", tightening_cash.get("I3101"))
print("평시 사전현금여력(%):", normal_cash.get("I3101"))


재무_원자재_병합패널.csv: 240행 갱신, 현금여력 매칭 안 된 업종 0개

 
평시_재무_원자재_병합패널.csv: 200행 갱신, 현금여력 매칭 안 된 업종 0개 

샘플(의약 I3101):
긴축기 사전현금여력(%): 207.2504989851384
평시 사전현금여력(%): 175.64019334393305


**결과**: 40업종 전부 계산 성공(실패 업종 없음), 두 병합패널(240행/200행) 전부 `사전현금여력(%)`·`Post×사전현금여력` 열 추가 완료. 의약(I3101) 기준 긴축기 현금여력≈207.25%, 평시 현금여력≈175.64%.

**해석**: 이제 두 병합패널에는 (노출도, 현금여력, 원자재충격) 세 종류의 변수가 다 갖춰졌다. 다음 단계인 사전추세 검정(`02_사전추세_검정.ipynb`)은 이 두 파일을 그대로 읽어서 진행한다.
